# r/soccer analytical pipeline on Google Colab (GPU)

**FIFA Communities chapter — `code/06-analysis-pipeline.py` run on a Colab GPU.** Orchestration only: every cell calls
the same command-line program that ran locally in the session-05 dry run; nothing analytical lives in this notebook.
Design: `docs/session-04_2026-10-22/01_analytical-pipeline-design.md` (v1.2; §10 for Colab). Approvals and records:
`docs/session-05_2026-10-23/` (implementation, dry run) and `docs/session-06_2026-10-24/` (two-step taxonomy protocol D5,
this test batch).

**Runtime:** *Runtime → Change runtime type → GPU*, A100 40 GB preferred (L4 acceptable; T4 works in float16, ≈ 3× slower),
high-RAM. In VS Code with the Colab extension: open this notebook, choose *New Colab Server → GPU*.

**Before the first run, once:** the folder `data/yearly datasets/` (17 CSV files) must be on your Google Drive as
`MyDrive/fifa-reddit-communities/data/yearly datasets/`. Results (`results/analysis/`, `data/analysis/`), the embedding
cache and the model weights (`.hf-cache`) are kept on Drive, so a disconnect costs nothing that is finished. Every stage's
output is also appended to a **session log on Drive** (`MyDrive/fifa-reddit-communities/logs/colab_<date>.log`), which is
what the reviewer reads afterwards.

**Two passes.** *Pass 1 — smoke test on bin 1* (≈ 15–20 min of GPU time): every stage once, labelling only the 12 largest
topics, with a check cell after each stage that compares with the local dry run (92 topics, 35.9 % raw outliers).
*Pass 2 — the real run*: bin 1 in full (all topics labelled; the **two-step taxonomy**: category definitions, then one
assignment debate per topic), then — in the next session — the gates and bins 2–9. **The session-06 test batch stops after
section 3.1** and hands the outputs back with the zip of section 5. Order of operations (design §3): bins → embed + topics →
**label** → *co-author review* (`topic_overrides.csv`) → **taxonomy** → *review* (`category_overrides.csv`,
`topic_category_overrides.csv`) → calibration of the outlier threshold `T` → `reassign` → sentiment → collect → copy back →
S6 locally in R.

## 0. Environment (run cells 0.1–0.4 every session; ≈ 3 min)

In [ ]:
# 0.1 Google Drive. This cell tests whether drive.mount works from this kernel (the VS Code Colab extension was not
#     documented on this point); the fallback downloads a shared Drive folder with gdown. It also opens the session log.
import os, subprocess, sys, datetime
DRIVE = '/content/drive/MyDrive/fifa-reddit-communities'    # adjust if you used another folder name
SHARED_DATA_URL = ''   # fallback only: a Drive *folder* link to 'data/yearly datasets' shared with the account
try:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.isdir(f'{DRIVE}/data/yearly datasets'), f'{DRIVE}/data/yearly datasets not found on Drive'
    n = len([f for f in os.listdir(f'{DRIVE}/data/yearly datasets') if f.endswith('.csv')])
    print('Drive mounted; yearly files on Drive:', n)
    assert n == 17, 'expected 17 yearly files'
except Exception as e:
    print('drive.mount failed or folder missing:', e)
    if not SHARED_DATA_URL:
        raise SystemExit('Set SHARED_DATA_URL to a shared Drive folder link and re-run this cell')
    os.makedirs(f'{DRIVE}/data', exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run(['gdown', '--folder', SHARED_DATA_URL, '-O', f'{DRIVE}/data/yearly datasets'], check=True)
# session log on Drive: everything printed through slog() and every stage's output lands here as well
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
SESSION_LOG = f'{DRIVE}/logs/colab_{datetime.datetime.utcnow():%Y-%m-%d_%H%M}.log'
def slog(*parts):
    line = ' '.join(str(p) for p in parts)
    print(line, flush=True)
    with open(SESSION_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n'); f.flush()
slog(f'=== Colab session log {SESSION_LOG} (UTC {datetime.datetime.utcnow():%Y-%m-%d %H:%M}) ===')

In [ ]:
# 0.2 Code from GitHub (public repository), data copied from Drive to the local SSD (Drive I/O is slow for large CSVs),
#     results / caches / weights symlinked to Drive so they persist across sessions.
import os, subprocess, shutil
REPO = '/content/fifa-reddit-communities'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/alannemi/fifa-reddit-communities', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO)
slog('repository at commit', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())
os.makedirs('data', exist_ok=True)
if not os.path.isdir('data/yearly datasets'):
    shutil.copytree(f'{DRIVE}/data/yearly datasets', 'data/yearly datasets')
files = sorted(f for f in os.listdir('data/yearly datasets') if f.endswith('.csv'))
assert len(files) == 17, files
slog('yearly files on local SSD:', len(files), '| total GB:', round(sum(os.path.getsize(f'data/yearly datasets/{f}') for f in files) / 1e9, 2))
for local, remote in [('results/analysis', f'{DRIVE}/results/analysis'), ('data/analysis', f'{DRIVE}/data/analysis'),
                      ('results/topic-models/embeddings', f'{DRIVE}/results/topic-models/embeddings'), ('.hf-cache', f'{DRIVE}/.hf-cache')]:
    os.makedirs(remote, exist_ok=True); os.makedirs(os.path.dirname(local) or '.', exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    os.symlink(remote, local)
os.environ['HF_HOME'] = f'{REPO}/.hf-cache'; os.environ['TOKENIZERS_PARALLELISM'] = 'false'
slog('working in', os.getcwd(), '| results/analysis ->', os.path.realpath('results/analysis'))

In [ ]:
# 0.3 Environment: install ONLY the pinned packages on top of Colab's RAPIDS stack (review finding R8), then prove that
#     cuML, CUDA and the pins are intact. If `pip check` reports a broken requirement, fix that one package explicitly
#     (do not re-run a plain `pip install -r`). Everything is written to the session log.
import subprocess, sys
inst = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-r', 'code/requirements-colab.txt'], capture_output=True, text=True)
slog('pip install (--no-deps) exit', inst.returncode, (inst.stdout + inst.stderr).strip()[-2000:])
assert inst.returncode == 0, 'pip install failed — read the output above'
chk = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
slog('pip check:', chk.stdout.strip() or 'OK')
import torch, cuml, bertopic, umap, hdbscan, sentence_transformers, transformers, numpy, sklearn, pandas
assert torch.cuda.is_available(), 'no CUDA device — change the runtime type to GPU'
slog('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| cuml', cuml.__version__)
slog('bertopic', bertopic.__version__, '| umap', umap.__version__, '| hdbscan', hdbscan.__version__,
     '| sentence-transformers', sentence_transformers.__version__, '| transformers', transformers.__version__,
     '| numpy', numpy.__version__, '| sklearn', sklearn.__version__, '| pandas', pandas.__version__, '| python', sys.version.split()[0])
assert bertopic.__version__ == '0.17.4' and umap.__version__ == '0.5.12' and hdbscan.__version__ == '0.8.44', 'pins not honoured'
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'], capture_output=True, text=True)
slog(smi.stdout.strip())

In [ ]:
# 0.4 Helpers. stage() runs one stage of the program (accel=True prefixes `python -m cuml.accel` for the GPU UMAP/HDBSCAN),
#     streams its output line by line to the notebook AND to the session log (child unbuffered, progress bars off;
#     review finding R33) and raises on a non-zero exit. cfg() reads a bin's config.json; taxonomy_summary() prints the
#     two-step taxonomy's outcome (definitions debate, assignment debates, repair, salvage).
import subprocess, sys, shlex, time, json, os
from collections import Counter
import pandas as pd
ENV = {**os.environ, 'PYTHONUNBUFFERED': '1', 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TQDM_DISABLE': '1'}
def run_logged(cmd):
    slog('>>>', ' '.join(cmd)); t0 = time.time()
    with open(SESSION_LOG, 'a', encoding='utf-8') as f, subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                           text=True, env=ENV, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True); f.write(line); f.flush()
    slog(f'<<< exit {p.returncode} after {(time.time() - t0) / 60:.1f} min')
    if p.returncode != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}): {" ".join(cmd)}')
def stage(args, accel=False):
    run_logged([sys.executable, '-u'] + (['-m', 'cuml.accel'] if accel else []) + ['code/06-analysis-pipeline.py'] + shlex.split(args))
def cfg(b):
    return json.load(open(f'results/analysis/bin-{b}/config.json'))
def unit_checks():
    run_logged([sys.executable, '-u', 'code/tests/test_pipeline_units.py'])
def taxonomy_summary(b):
    tj = json.load(open(f'results/analysis/bin-{b}/taxonomy.json')); c = pd.read_csv(f'results/analysis/bin-{b}/categories.csv')
    items = tj.get('assignment_items') or []; ca = Counter(tj['category_assignment'].values())
    slog(f"taxonomy bin {b}: protocol {tj['protocol']} | definitions debate: {tj['agreement_mode']}, {tj['round_iterations']} round(s), needs_review {tj['needs_review']}")
    slog(f"  categories defined {len(tj.get('categories_defined') or [])} -> final {len(c)} | dropped (no topic assigned): {tj.get('categories_dropped_empty')} | cleanup: {tj.get('definitions_cleanup')}")
    slog(f"  assignment items {len(items)} | modes {dict(Counter(i['agreement_mode'] for i in items))} | items contested {sum(1 for i in items if i.get('n_contested'))}")
    slog(f"  topics assigned by the agents {ca.get('agent', 0)}, placed by the repair {ca.get('repaired', 0)}, unplaced {ca.get('unplaced', 0)} | answers salvaged {tj['n_answers_salvaged']}")
    if len(c):
        slog(c[['category_id', 'name', 'tone_category', 'n_topics', 'n_posts', 'share']].to_string(index=False))
    return tj, c
unit_checks()          # ~1 min, no model needed — proves the code imports and behaves on this machine

## 1. S0 — period bins (every post; Σ must equal 1,302,021 and the nine counts the design table)

In [ ]:
stage('--stage bins')
s = pd.read_csv('data/analysis/period_bins_summary.csv')
slog(s[['period_bin', 'period_type', 'n_posts', 'expected_n_design', 'min_cluster_size_start']].to_string(index=False))
assert (s.n_posts == s.expected_n_design).all() and s.n_posts.sum() == 1302021, 'bin sizes differ from the design table'

## 2. Pass 1 — smoke test on bin 1 (≈ 15–20 min of GPU time)
Every stage once on bin 1, with labelling limited to the 12 largest topics. Each check cell compares with the local dry run
of 2026-10-23 (MiniLM, CPU): 10,002 posts, 9,809 modelled, **92 topics at min_cluster_size 25, 35.9 % raw outliers**;
12 labelled topics, 1 needing review. With mpnet embeddings the numbers will differ somewhat — the check is that the stages
run on the GPU backend and the orders of magnitude match. The taxonomy runs the two-step protocol over the 12 labelled
topics (definitions, then one assignment debate per topic). Outputs go to the real `results/analysis/bin-1/`; Pass 2
resumes from them (labelling continues from `labels.jsonl`; the taxonomy is re-run with `--force`).

In [ ]:
stage('--stage embed --bin 1')
stage('--stage topics --bin 1', accel=True)
c = cfg(1)['topics']
slog(json.dumps({k: c[k] for k in ['n_documents', 'n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1))
slog('backend:', c['backend']); slog('guard:', c['guard'])
assert c['backend']['backend'] == 'cuml', 'UMAP/HDBSCAN did not run on cuML — check that the command was prefixed with python -m cuml.accel'
assert c['guard']['converged'], 'topic-count guard did not reach 50–100 topics: see guard trials above (design §6.2, dry-run finding D1)'
slog('calibration candidates (T, n reassigned):', [(r['threshold'], r['n_reassigned_at_T']) for r in c['calibration']])

In [ ]:
stage('--stage label --bin 1 --max-topics 12')
t = pd.read_csv('results/analysis/bin-1/topics_labelled.csv')
slog(t[['topic_id', 'name', 'tone_category', 'round_iterations', 'agreement_mode', 'needs_review']].to_string(index=False))
l = cfg(1)['label']; slog('llm:', l['llm_model'], l['llm_revision'], l['dtype'], 'batch', l['batch_size'], '| prompt', l['prompt_version'], '| stats', l['llm_stats'], '| min', l['elapsed_min'])
assert len(t) == 12 and l['llm_stats']['truncated'] == 0

In [ ]:
stage('--stage taxonomy --bin 1')          # two-step protocol over the 12 labelled topics (config.TAXONOMY_PROTOCOL)
tj, c = taxonomy_summary(1)
assert tj['protocol'] == 'two_step', 'the taxonomy did not run the two-step protocol — check config.py / --taxonomy-protocol'
assert len(c) >= 1, 'no category produced — read taxonomy.json'
slog('taxonomy config:', {k: cfg(1)['taxonomy'].get(k) for k in ['n_categories_defined', 'n_categories', 'n_items', 'assignment_agreement_modes', 'n_repaired', 'n_unplaced', 'n_answers_salvaged', 'elapsed_min']})

In [ ]:
stage('--stage sentiment --years 2010')      # the 2010 file only, for the smoke test; Pass 2 scores every year
stage('--stage collect --allow-partial')
side = json.load(open('data/analysis/post_variables.json'))
slog(json.dumps({k: side[k] for k in ['n_posts', 'bins_complete', 'complete', 'n_with_sentiment', 'n_with_topic', 'n_with_category', 'errors']}, indent=1))
assert side['n_posts'] == 1302021 and side['bins_complete'] == [1] and not side['errors']
slog('Pass 1 complete — open results/analysis/bin-1/report.html (on Drive) and compare with the local dry run.')

## 3. Pass 2 — the real run
### 3.1 Bin 1 in full — gate 1 (design §12.1) and gate 4 (§12.4) — **the session-06 test batch ends here**
Resumes from Pass 1: the remaining topics are labelled; then the two-step taxonomy over all topics (decision D5,
`docs/session-06_2026-10-24/00_approvals.md`). Read `report.html` (2010 themes: vuvuzelas, Jabulani, Lampard's goal, Suárez's
handball, Spain's win, Paul the octopus) and the label summary (`needs_review` ≤ 15 %, the `agreement_mode` and round
distributions). The taxonomy summary shows the definitions debate, the assignment debates, how many topics the repair had to
place and how many answers the JSON recovery touched.

In [ ]:
stage('--stage label --bin 1')                 # resumable; re-run the cell after a disconnect
t = pd.read_csv('results/analysis/bin-1/topics_labelled.csv')
slog(f'{len(t)} topics | needs_review {t.needs_review.sum()} ({t.needs_review.mean():.0%}) | modes {t.agreement_mode.value_counts().to_dict()} | rounds {t.round_iterations.value_counts().sort_index().to_dict()}')
if t.needs_review.mean() > 0.15:
    slog('WARNING: needs_review above the 15 % gate (design §12.4) — a decision for the author before bins 2–9')
# S3b over ALL topics: the two-step protocol (decision D5, 2026-10-24). --force because Pass 1 wrote a 12-topic taxonomy;
# it also removes the step-1 checkpoint, so both steps run again over the full topic set.
stage('--stage taxonomy --bin 1 --force')
tj, c = taxonomy_summary(1)
n_unplaced = sum(v == 'unplaced' for v in tj['category_assignment'].values())
if n_unplaced or tj['needs_review']:
    slog(f'WARNING: unplaced {n_unplaced}, definitions needs_review {tj["needs_review"]} — read taxonomy.json before the review')
stage('--stage collect --allow-partial')
slog('=== STOP POINT of the session-06 test batch: run the hand-back cell in section 5 ===')

### Stop point of the session-06 test batch
Everything above is the agreed test batch (Pass 1 + the full bin 1). Run the **hand-back cell in section 5**, which zips
`results/analysis/bin-1/` (without the regenerable `model/`), the aggregated tables, `data/analysis/` and the session log into
`MyDrive/fifa-reddit-communities/handback/`. Then either download the zip into the repository (unzip at the repository root)
or let the reviewer read it from Drive. The cells below (comparisons, large-bin gate, `T`, bins 2–9, sentiment for all years,
the complete collect) belong to the next session.

### 3.2 Optional comparison on bin 1 (design §12.4, F17: seeded sampling) — into its own folder, so bin 1's real outputs stay clean
The Qwen3-4B comparison (design F9) was dropped by the author on 2026-10-24: the pipeline stays with Qwen3-1.7B.

In [ ]:
RUN_COMPARISONS = False
if RUN_COMPARISONS:
    import shutil
    for name, extra in [('cmp-seeded', '--sampling seeded')]:
        root = f'/content/{name}'
        os.makedirs(f'{root}/results/analysis', exist_ok=True); os.makedirs(f'{root}/data/analysis', exist_ok=True)
        if not os.path.isdir(f'{root}/results/analysis/bin-1'):
            shutil.copytree('results/analysis/bin-1', f'{root}/results/analysis/bin-1', ignore=shutil.ignore_patterns('labels*', 'topics_labelled.csv', 'taxonomy*.json', 'categories.csv', 'topic_categories.csv', '*_overrides.csv'))
            shutil.copy('data/analysis/period_bins.csv', f'{root}/data/analysis/')
        stage(f'--stage label --bin 1 {extra} --out-root {root}')
        t2 = pd.read_csv(f'{root}/results/analysis/bin-1/topics_labelled.csv')
        slog(name, '| needs_review', t2.needs_review.mean().round(3), '| modes', t2.agreement_mode.value_counts().to_dict())

### 3.3 Large-bin gate (design §12.2): bin 4 (331k posts) or bin 6 (306k)
Raw outlier share ≤ 45 %, guard converged within four trials, minutes on the GPU. If the gate fails, revisit `HDBSCAN_MIN_SAMPLES`
and/or `TOPIC_COUNT_RANGE` (a decision for the design's §14) before any other bin.

In [ ]:
GATE_BIN = 4
stage(f'--stage embed --bin {GATE_BIN}')
stage(f'--stage topics --bin {GATE_BIN}', accel=True)
c = cfg(GATE_BIN)['topics']
print(json.dumps({k: c[k] for k in ['n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1), '| guard', c['guard'], '| backend', c['backend']['backend'])
assert c['outlier_share_raw'] <= 0.45 and c['guard']['converged'], 'large-bin gate failed — see design §12.2'
stage(f'--stage label --bin {GATE_BIN}')
stage(f'--stage taxonomy --bin {GATE_BIN}')

### 3.4 Outlier threshold `T` (decision D3; gate §12.3)
`results/analysis/bin-N/outlier_calibration/` holds the quantile table and one 100-post judgement sample per candidate threshold
(`belongs` column for the co-author), for bin 1 and the large bin. Once `T` is chosen, record it in `config.py`
(`OUTLIER_THRESHOLD`) and in the design's decisions log, then apply it here (seconds per bin; never changes `topic_id_raw`,
keywords or labels — only sizes, prevalence and the report).

In [ ]:
T = None                        # e.g. 0.55 after the co-author check
if T is not None:
    for b in sorted({1, GATE_BIN}):
        stage(f'--stage reassign --bin {b} --outlier-threshold {T}')

### 3.5 Bins 2–9 (after the reviews of bin 1 and the gate bin): embed + topics → label → *review* → taxonomy → *review* → reassign

In [ ]:
BINS = [b for b in range(2, 10) if b != GATE_BIN]
for b in BINS:
    stage(f'--stage embed --bin {b}')
    stage(f'--stage topics --bin {b}', accel=True)
    c = cfg(b)['topics']; print(f"bin {b}: {c['n_topics_raw']} topics, raw outliers {c['outlier_share_raw']:.1%}, backend {c['backend']['backend']}, guard converged {c['guard']['converged']}")
for b in BINS:
    stage(f'--stage label --bin {b}')
# --- review pause: topic_overrides.csv per bin --- then:
# for b in BINS: stage(f'--stage taxonomy --bin {b}')
# --- review pause: category_overrides.csv per bin --- then reassign with the fixed T:
# for b in BINS: stage(f'--stage reassign --bin {b} --outlier-threshold {T}')

## 4. S4 sentiment (every year, CPU, ≈ 15 min) and S5 collect

In [ ]:
stage('--stage sentiment --force')
stage('--stage collect')          # without --allow-partial: must end with complete = true
side = json.load(open('data/analysis/post_variables.json'))
print(json.dumps({k: v for k, v in side.items() if k != 'versions'}, indent=1))
assert side['complete'], 'collection not complete — see bins_state above'

## 5. Hand-back, copy back and finish locally
`results/analysis/` and `data/analysis/` are symlinked to Drive, so they are already there. The cell below zips what the
reviewer needs (`results/analysis/bin-1/` without `model/`, the aggregated tables, `data/analysis/*.csv|json`, the session
log) into `MyDrive/fifa-reddit-communities/handback/`. Download the zip and unzip it at the repository root (`results/analysis/`
is tracked in git; `data/analysis/` goes to the shared data folder). After the full run, stage S6 runs locally:
```bash
Rscript code/07-tag-corpus.R && Rscript code/08-validate-analysis.R
```
and codebook v3.0 is finalised (`docs/session-05_2026-10-23/03_runbook-session-06.md`).

In [ ]:
# hand-back zip + inventory
import zipfile, datetime
stamp = datetime.datetime.utcnow().strftime('%Y-%m-%d_%H%M')
os.makedirs(f'{DRIVE}/handback', exist_ok=True)
zpath = f'{DRIVE}/handback/bin-1_{stamp}.zip'
n_files = 0
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for dirpath, dirs, files in os.walk('results/analysis/bin-1'):
        dirs[:] = [d for d in dirs if d != 'model']                      # the BERTopic model folder is regenerable
        for fn in files:
            z.write(os.path.join(dirpath, fn), arcname=os.path.join(dirpath, fn)); n_files += 1
    for fn in os.listdir('results/analysis'):                            # the aggregated tables at the top level
        if fn.endswith('.csv'):
            z.write(os.path.join('results/analysis', fn), arcname=os.path.join('results/analysis', fn)); n_files += 1
    for fn in os.listdir('data/analysis'):
        if fn.endswith(('.csv', '.json', '.txt')):
            z.write(os.path.join('data/analysis', fn), arcname=os.path.join('data/analysis', fn)); n_files += 1
    z.write(SESSION_LOG, arcname='logs/' + os.path.basename(SESSION_LOG)); n_files += 1
slog('hand-back zip:', zpath, '|', n_files, 'files |', round(os.path.getsize(zpath) / 1e6, 1), 'MB')
for root in ['results/analysis', 'data/analysis']:
    for dirpath, dirs, files in os.walk(root):
        if dirpath.count('/') <= 2:
            slog(dirpath, len(files), 'files')